# Stage 1 - Preprocessing & Validation Setup

This notebook contains:
1. **Environment & Paths**: Verify working directories and imports
2. **Dynamic Random Sampling**: Inspect random rows from the multi-million record TSV files instantly
3. **Text & Address Normalization**: Test name suffix stripping, address expansion, and French accents
4. **Local Validation Split**: Create a fast, balanced 10,000-query validation slice with ground truth
5. **Validation Verification**: Verify singleton ratio and entity matching structures
6. **Full Dataset Preprocessing**: Batch-process all train and test files into `dataset/preprocessed/`

In [1]:
# 1. Environment & Path Setup
import os
import sys
import random
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"Root Directory : {ROOT_DIR}")

Python Version : 3.14.3
Pandas Version : 3.0.6
Root Directory : c:\Users\satya\Downloads\amazon


In [2]:
# 2. Fast Dynamic Random Sampling from Dataset
def get_random_sample(filepath, n=5):
    """Pick n random rows from a large TSV file instantly using random byte seeks."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip().split("\t")
        file_size = os.path.getsize(filepath)
        rows = []
        for _ in range(n * 5):
            pos = random.randint(0, max(0, file_size - 2000))
            f.seek(pos)
            f.readline()  # discard partial line
            line = f.readline().strip()
            if line:
                parts = line.split("\t")
                if len(parts) == len(header):
                    rows.append(parts)
            if len(rows) >= n:
                break
    return pd.DataFrame(rows[:n], columns=header)

DATA_DIR = "../student_resource/dataset/train"

print("Random Source 1 sample (changes every run):")
s1_sample = get_random_sample(f"{DATA_DIR}/train_source1.tsv", n=5)
display(s1_sample)

print("\nRandom Ground Truth sample (changes every run):")
gt_sample = get_random_sample(f"{DATA_DIR}/train_ground_truth.tsv", n=5)
display(gt_sample)

Random Source 1 sample (changes every run):


,entity_id,business_name,business_address,country
0,S1-211370935,North Solana LLC,"2306 Bestwick Road, Greensboro, NC",US
1,S1-486482259,Sapphire P.C.,"2161 Nolensville Pike, Unit 107, Nashville, TN",US
2,S1-80697540,Northwind Earth LP,"6334 Peppermint Lane, Crosby, TX",US
3,S1-707929597,Wiles Semiconductor Inc,"504 Oak Street, Mayfield, KY",US
4,S1-961351936,Sree It Private Limited,"Plot No.B-62, Malegaon Midc, Sinnar R-I Sectio...",India



Random Ground Truth sample (changes every run):


,source1_entity_id,matched_entity_ids
0,S1-288395412,"S2-893640034,S3-697414040"
1,S1-47708709,"S2-802655753,S2-521202799,S3-998829038,S3-9097..."
2,S1-220164954,"S2-403766865,S3-606351862,S3-121345227,S3-3781..."
3,S1-98540031,"S2-632703241,S3-202801074,S3-156278476"
4,S1-119682882,"S2-728864376,S3-527539385,S3-790940984,S3-9208..."


In [3]:
# 3. Test Preprocessing Pipeline on Random Real Records + France Test Case
from src.preprocessing import preprocess_record

# Real records sampled dynamically from Cell 2
real_records = s1_sample.to_dict("records")

# Include a French test case (from test set distribution)
france_sample = {
    "entity_id": "TEST-FR-01",
    "business_name": "Société Générale S.A.R.L.",
    "business_address": "12 Bd Haussmann, 75009 Paris",
    "country": "France",
}
all_samples = real_records + [france_sample]

print(f"=== Preprocessing {len(all_samples)} Records (Dynamic Real Samples + French Test Case) ===\n")
for s in all_samples:
    cleaned = preprocess_record(s)
    print(f"ID: {s['entity_id']} ({s.get('country', '')})")
    print(f"  Original : {s.get('business_name', '')} | {s.get('business_address', '')}")
    print(f"  Cleaned  : {cleaned['clean_name']} [Core: {cleaned['core_name']}] (Suffix: '{cleaned['legal_suffix']}')")
    print(f"  Address  : {cleaned['clean_address']}")
    print(f"  Postal   : {cleaned['postal_code']}")
    print("-" * 80)

=== Preprocessing 6 Records (Dynamic Real Samples + French Test Case) ===

ID: S1-211370935 (US)
  Original : North Solana LLC | 2306 Bestwick Road, Greensboro, NC
  Cleaned  : north solana llc [Core: north solana] (Suffix: 'llc')
  Address  : 2306 bestwick road greensboro nc
  Postal   : 
--------------------------------------------------------------------------------
ID: S1-486482259 (US)
  Original : Sapphire P.C. | 2161 Nolensville Pike, Unit 107, Nashville, TN
  Cleaned  : sapphire pc [Core: sapphire pc] (Suffix: '')
  Address  : 2161 nolensville pike unit 107 nashville tn
  Postal   : 
--------------------------------------------------------------------------------
ID: S1-80697540 (US)
  Original : Northwind Earth LP | 6334 Peppermint Lane, Crosby, TX
  Cleaned  : northwind earth lp [Core: northwind earth] (Suffix: 'lp')
  Address  : 6334 peppermint lane crosby tx
  Postal   : 
--------------------------------------------------------------------------------
ID: S1-707929597 (US)


In [4]:
# 4. Generate Fast Stratified Local Validation Split (10,000 S1 queries)
!python ../scripts/create_val_split.py --data-dir ../student_resource/dataset/train --out-dir ../dataset/val_split --num-s1 10000 --num-distractors 30000

Loading ground truth from ../student_resource/dataset/train/train_ground_truth.tsv ...
Selected 10,000 Source 1 entities.
Ground truth requires 16,687 S2 matches and 17,945 S3 matches.
Saved val_ground_truth.tsv
Filtering Source 1...
Saved val_source1.tsv with 10,000 rows.
Filtering Source 2...
Saved val_source2.tsv with 526,126 rows.
Filtering Source 3...
Saved val_source3.tsv with 557,315 rows.

Validation split creation complete in: ../dataset/val_split


In [5]:
# 5. Verify the Generated Validation Dataset
val_s1 = pd.read_csv("../dataset/val_split/val_source1.tsv", sep="\t")
val_gt = pd.read_csv("../dataset/val_split/val_ground_truth.tsv", sep="\t")

singletons = (val_gt["matched_entity_ids"].isna() | (val_gt["matched_entity_ids"] == "")).sum()
print(f"Validation Source 1 rows    : {len(val_s1):,}")
print(f"Validation Ground Truth rows : {len(val_gt):,}")
print(f"Singletons (no match)        : {singletons:,} ({singletons / len(val_gt) * 100:.1f}%)")
display(val_s1.head(3))
display(val_gt.head(3))

Validation Source 1 rows    : 10,000
Validation Ground Truth rows : 10,000
Singletons (no match)        : 560 (5.6%)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-68456921,Dynamic Hospitality Private Limited,"2Nd Floor, Trivedi Bhawan, Pghpl Group Campus,...",India
2,S1-874413333,Primary Care Classic Associates LLC,"James City County, VA, 3005 London Company Way",US


,source1_entity_id,matched_entity_ids
0,S1-508337570,"S2-582025035,S3-529082874"
1,S1-878749336,"S2-790690416,S2-733913751,S3-932487174"
2,S1-49527993,"S2-860744957,S2-39902340,S2-237585210,S3-82327..."


### 6. Preprocess Complete Dataset (Train & Test)

Runs streaming chunked normalization across all 6 files:
- `train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`
- `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`

Outputs are saved in `dataset/preprocessed/` with clean names, core names, legal suffixes, standardized addresses, and postal codes.

In [6]:
# Run high-throughput streaming preprocessing across the complete dataset
!python ../scripts/preprocess_dataset.py --input-dir ../student_resource/dataset --output-dir ../dataset/preprocessed --split all --chunksize 150000

=== Starting Preprocessing (Split: all) ===
Input base : ../student_resource/dataset
Output base: ../dataset/preprocessed
Total files: 6

Processing train_source1.tsv -> train_source1.tsv ...
Completed 2,206,821 records in 270.7s (8,153 records/s).

Processing train_source2.tsv -> train_source2.tsv ...
Completed 5,034,616 records in 438.3s (11,487 records/s).

Processing train_source3.tsv -> train_source3.tsv ...
Completed 5,285,603 records in 639.9s (8,260 records/s).

Processing test_source1.tsv -> test_source1.tsv ...
Completed 1,732,544 records in 253.1s (6,845 records/s).

Processing test_source2.tsv -> test_source2.tsv ...
Completed 4,887,273 records in 645.8s (7,568 records/s).

Processing test_source3.tsv -> test_source3.tsv ...
Completed 5,082,316 records in 515.4s (9,862 records/s).

All tasks finished in 46.1 minutes!



train_source1.tsv: 0chunk [00:00, ?chunk/s]
train_source1.tsv: 1chunk [00:21, 21.71s/chunk]
train_source1.tsv: 2chunk [00:39, 19.13s/chunk]
train_source1.tsv: 3chunk [01:00, 20.24s/chunk]
train_source1.tsv: 4chunk [01:25, 21.93s/chunk]
train_source1.tsv: 5chunk [01:48, 22.54s/chunk]
train_source1.tsv: 6chunk [02:06, 20.83s/chunk]
train_source1.tsv: 7chunk [02:24, 20.02s/chunk]
train_source1.tsv: 8chunk [02:46, 20.72s/chunk]
train_source1.tsv: 9chunk [03:10, 21.60s/chunk]
train_source1.tsv: 10chunk [03:26, 19.78s/chunk]
train_source1.tsv: 11chunk [03:39, 17.90s/chunk]
train_source1.tsv: 12chunk [03:53, 16.55s/chunk]
train_source1.tsv: 13chunk [04:06, 15.68s/chunk]
train_source1.tsv: 14chunk [04:20, 15.06s/chunk]
train_source1.tsv: 15chunk [04:30, 13.51s/chunk]
train_source1.tsv: 15chunk [04:30, 18.03s/chunk]

train_source2.tsv: 0chunk [00:00, ?chunk/s]
train_source2.tsv: 1chunk [00:16, 16.16s/chunk]
train_source2.tsv: 2chunk [00:34, 17.45s/chunk]
train_source2.tsv: 3chunk [00:46, 15.16